<a href="https://colab.research.google.com/github/zencolab/WhatDreamsCost-ComfyUI/blob/main/IndexTTS_2_5_Colab_L4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IndexTTS 2.5：音频-only口型安全修复

**从上到下运行。**一次上传两个文件：视频和已获授权的参考声音。

为避免声音与口型错位：
- 不再重新生成或强行压缩整句；
- 只局部替换音节数量相同的错词，句子其余部分保持原声；
- 新增或删除音节、`ADDITIONS`新增台词会自动跳过，因为原画面没有对应嘴部动作；
- 其他人物声音、停顿、语气、背景音乐和音效保持不变。

最终输出 `/content/video_new_voice.mp4`。


In [ ]:
# 步骤1：确认使用 Colab Pro 的 NVIDIA L4 GPU
import os, subprocess
assert os.path.exists('/content'), '请在 Google Colab 中运行。'
gpu = subprocess.check_output(
    ['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], text=True
).strip()
print('检测到：', gpu)
assert 'L4' in gpu, f'当前不是L4：{gpu}'


In [ ]:
# 步骤2：安装免费字幕识别、音轨分离和IndexTTS
%cd /content
!python -m pip install -q -U uv huggingface_hub hf_xet pysrt faster-whisper demucs

from pathlib import Path
from huggingface_hub import snapshot_download
import subprocess, urllib.request

repo = Path('/content/index-tts')
if not repo.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/index-tts/index-tts.git',str(repo)], check=True)
else:
    subprocess.run(['git','-C',str(repo),'fetch','--depth','1','origin','main'], check=True)
    subprocess.run(['git','-C',str(repo),'reset','--hard','origin/main'], check=True)

%cd /content/index-tts
!uv sync --extra webui
!uv pip install --python /content/index-tts/.venv/bin/python pysrt pydub

model_dir = Path('/content/index-tts/checkpoints')
model_dir.mkdir(parents=True, exist_ok=True)
snapshot_download(repo_id='IndexTeam/IndexTTS-2.5', local_dir=str(model_dir))
assert (model_dir/'config.yaml').exists(), 'IndexTTS模型下载不完整。'

base = 'https://raw.githubusercontent.com/zencolab/WhatDreamsCost-ComfyUI/main/'
urllib.request.urlretrieve(base+'auto_replace_voice.py', '/content/index-tts/auto_replace_voice.py')
urllib.request.urlretrieve(base+'generate_srt.py', '/content/generate_srt.py')
urllib.request.urlretrieve(base+'prepare_audio.py', '/content/prepare_audio.py')
print('✅ 安装和下载完成')


## 步骤3：上传视频和参考声音

同时选择视频和参考声音。程序会免费生成SRT，并分离原人声、音乐和音效。


In [ ]:
# 步骤3：上传、生成字幕并分离音轨
from google.colab import files
from pathlib import Path
import subprocess, sys, pysrt

uploaded = files.upload()
names = list(uploaded)
def one(exts, label):
    matches = [n for n in names if Path(n).suffix.lower() in exts]
    if len(matches) != 1:
        raise ValueError(f'需要且只能上传一个{label}；识别到：{matches}')
    return matches[0]

video = one({'.mp4','.mov','.mkv','.webm'}, '视频')
voice = one({'.wav','.mp3','.m4a','.flac','.ogg'}, '参考声音')
Path('/content/video.mp4').write_bytes(uploaded[video])
voice_source = Path('/content/uploaded_voice'+Path(voice).suffix.lower())
voice_source.write_bytes(uploaded[voice])
subprocess.run([
    'ffmpeg','-y','-loglevel','error','-i',str(voice_source),
    '-ac','1','-ar','24000','/content/voice.wav'
], check=True)

# 两个模型均在独立进程运行，完成后释放显存。
subprocess.run([
    sys.executable,'/content/generate_srt.py','--video','/content/video.mp4',
    '--output','/content/subtitles.srt','--model','large-v3','--language','zh'
], check=True)
subprocess.run([sys.executable,'/content/prepare_audio.py'], check=True)

print('✅ 字幕如下：')
for row in pysrt.open('/content/subtitles.srt', encoding='utf-8'):
    print(f'#{row.index}  {row.start} --> {row.end}  |  {row.text.replace(chr(10), " ")}')


## 步骤4：填写等音节错词修正

**只改音频时，新增或删除音节无法与原口型同步。**程序只局部替换音节数量相同的错词；不符合条件的修正会明确提示并跳过，不再生成整句错位声音。

- 可修复：`我一共卯了八盒` → `我一共买了八盒`（音节数量相同）
- 会跳过：`你还吃` → `你还吃啊`（新增了“啊”）
- 正常结束用中文句号：`。`；短停顿用：`，`；较长停顿用：`……`


In [ ]:
# 只填写音节数量不变的错词；新增/删减字会自动跳过。
CORRECTIONS = {
    # 2: '我一共买了八盒',  # ✅ 与原句音节数量相同
    # 1: '你还吃啊！？',    # ❌ 新增“啊”，只改音频时会跳过
}

# 只改音频不能凭空增加口型，因此以下新增台词会跳过。
ADDITIONS = [
    # {'start':'00:00:12,500','end':'00:00:13,500','text':'新增台词。'},
]

import json
from pathlib import Path
Path('/content/dialogue_edits.json').write_text(
    json.dumps({'corrections':CORRECTIONS,'additions':ADDITIONS}, ensure_ascii=False, indent=2),
    encoding='utf-8'
)
print(f'✅ 修正{len(CORRECTIONS)}条，新增{len(ADDITIONS)}条')


In [ ]:
# 步骤5：仅局部生成等音节错词，并保留其余原始声音
# 每次运行都自动获取GitHub上的最新版处理程序。
import os, subprocess, urllib.request
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/zencolab/WhatDreamsCost-ComfyUI/main/auto_replace_voice.py',
    '/content/index-tts/auto_replace_voice.py'
)
env = os.environ.copy()
env['PYTHONPATH'] = '/content/index-tts' + os.pathsep + env.get('PYTHONPATH','')
subprocess.run([
    '/content/index-tts/.venv/bin/python','/content/index-tts/auto_replace_voice.py'
], cwd='/content/index-tts', env=env, check=True)
print('✅ 输出：/content/video_new_voice.mp4')


In [ ]:
# 步骤6：预览并下载成品
from IPython.display import Video, display
from google.colab import files
output = '/content/video_new_voice.mp4'
display(Video(output, embed=True))
files.download(output)
